In [1]:
import numpy as np
import rasterio
from pathlib import Path
import geopandas as gpd
from rasterio.mask import mask
import json
from datetime import date
import calendar

# =========================================================
# PATHS
# =========================================================
ROOT = Path(
    "/home/root_1/Documents/CDL/repos/IDS-DRR_Heat/Heat-assam/data_extractor/era5_land"
)
quarter_dir  = ROOT / "data" / "quarterly_tiffs"
threshold_dir = ROOT / "data" / "percentile_tiffs"
output_dir   = ROOT / "data" / "monthly_heatday_scores"
geojson_path = Path(
    "/home/root_1/Documents/CDL/repos/IDS-DRR_Heat/Heat-assam/data_extractor/Maps/Geojson/assam_district_2024-11.geojson"
)
output_dir.mkdir(parents=True, exist_ok=True)

# =========================================================
# LOAD DISTRICT GEOMETRY
# =========================================================
gdf  = gpd.read_file(geojson_path)
geoms = json.loads(gdf.to_json())["features"]
geoms = [f["geometry"] for f in geoms]

# =========================================================
# LOAD SINGLE BAND RASTER
# =========================================================
def load_raster(path):
    with rasterio.open(path) as src:
        return src.read(1)

# =========================================================
# QUARTER → MONTH BAND OFFSETS
# =========================================================
def get_band_slice_for_month(year: int, month: int):
    """
    Return (band_start, band_end) — 1-based, inclusive — for the
    given month inside its quarterly TIF.

    ERA-5 Land quarterly TIFs are assumed to be stored as one band
    per day, ordered Jan-1 … Dec-31 within the quarter.

    Q1 → months 1-3  (Jan / Feb / Mar)
    Q2 → months 4-6  (Apr / May / Jun)
    Q3 → months 7-9  (Jul / Aug / Sep)
    Q4 → months 10-12 (Oct / Nov / Dec)
    """
    quarter_first_month = ((month - 1) // 3) * 3 + 1   # 1, 4, 7, or 10

    # Count days before this month within the quarter
    day_offset = 0
    for m in range(quarter_first_month, month):
        day_offset += calendar.monthrange(year, m)[1]

    days_in_month = calendar.monthrange(year, month)[1]

    band_start = day_offset + 1                  # 1-based
    band_end   = day_offset + days_in_month      # inclusive
    return band_start, band_end


# =========================================================
# MONTHLY LOOP
# =========================================================
years = [2023, 2024]

for year in years:
    print(f"\n================ YEAR {year} ================")

    for month in range(1, 13):
        print(f"\nProcessing {year}-{month:02d}")

        # -------------------------------------------------
        # SEASON / QUARTER MAPPING
        # -------------------------------------------------
        if month in [1, 2, 3]:
            season, q = "JFM", "Q1"
        elif month in [4, 5, 6]:
            season, q = "AMJ", "Q2"
        elif month in [7, 8, 9]:
            season, q = "JAS", "Q3"
        else:
            season, q = "OND", "Q4"

        hi_path = quarter_dir / f"HI_{year}_{q}.tif"
        if not hi_path.exists():
            print("SKIP: HI missing")
            continue

        # -------------------------------------------------
        # LOAD THRESHOLDS
        # -------------------------------------------------
        try:
            p80 = load_raster(threshold_dir / f"{season}_P80_1990_2023.tif")
            p88 = load_raster(threshold_dir / f"{season}_P88_1990_2023.tif")
            p95 = load_raster(threshold_dir / f"{season}_P95_1990_2023.tif")
            p99 = load_raster(threshold_dir / f"{season}_P99_1990_2023.tif")
        except Exception as e:
            print(f"SKIP: missing thresholds for {season} — {e}")
            continue

        # -------------------------------------------------
        # FIGURE OUT WHICH BANDS BELONG TO THIS MONTH
        # -------------------------------------------------
        band_start, band_end = get_band_slice_for_month(year, month)
        expected_days = calendar.monthrange(year, month)[1]

        print(f"  Quarter {q}, bands {band_start}–{band_end} "
              f"({expected_days} days)")

        # -------------------------------------------------
        # LOAD ONLY THIS MONTH'S BANDS FROM THE QUARTERLY TIF
        # -------------------------------------------------
        with rasterio.open(hi_path) as src:
            total_bands = src.count
            if band_end > total_bands:
                print(f"  WARN: TIF has {total_bands} bands but need up to "
                      f"{band_end}. Clamping.")
                band_end = total_bands

            band_indices = list(range(band_start, band_end + 1))  # 1-based
            hi   = src.read(band_indices)                          # (days, rows, cols)
            meta = src.meta.copy()

        days, rows, cols = hi.shape
        print(f"  Loaded {days} daily layers")

        # -------------------------------------------------
        # MONTHLY HEATDAY SCORE
        # -------------------------------------------------
        monthly = np.zeros((rows, cols), dtype=np.float32)

        for d in range(days):
            h     = hi[d]
            score = np.zeros((rows, cols), dtype=np.uint8)
            score[(h >= p80) & (h < p88)] = 1
            score[(h >= p88) & (h < p95)] = 2
            score[(h >= p95) & (h < p99)] = 3
            score[h >= p99]               = 4
            monthly += score

        # =========================================================
        # WRITE TEMP RASTER
        # =========================================================
        tmp_file = output_dir / f"_tmp_{year}_{month:02d}.tif"
        
        # Use -9999 as nodata sentinel — never a valid score (scores are 0–4*days)
        NODATA_VAL = -9999.0
        
        meta.update({"count": 1, "dtype": "float32", "nodata": NODATA_VAL})
        with rasterio.open(tmp_file, "w", **meta) as dst:
            dst.write(monthly, 1)

        # =========================================================
        # CLIP TO DISTRICT BOUNDARIES
        # =========================================================
        with rasterio.open(tmp_file) as src:
            out_image, out_transform = mask(
                src,
                geoms,
                crop=True,
                filled=True,
                nodata=NODATA_VAL,   # outside boundary → -9999
                all_touched=True,    # include edge pixels touched by boundary
            )
        
        out_image = out_image.astype("float32")
        
        # Build clean output: replace ONLY the boundary nodata with NaN
        # DO NOT touch zero-score pixels — score=0 is valid (no heatdays)
        out_final = out_image[0].copy()
        out_final[out_final == NODATA_VAL] = np.nan   # boundary → NaN
        # Score=0 stays as 0 — it means "pixel had no heatdays this month"

        out_meta = meta.copy()
        out_meta.update({
            "height"   : out_image.shape[1],
            "width"    : out_image.shape[2],
            "transform": out_transform,
            "count"    : 1,
            "nodata"   : np.nan,
        })

        # =========================================================
        # FINAL SAVE
        # =========================================================
        final_file = output_dir / f"HEATDAY_{year}_{month:02d}.tif"
        with rasterio.open(final_file, "w", **out_meta) as dst:
            dst.write(out_final, 1)

        tmp_file.unlink()
        print(f"  Saved: {final_file}  "
              f"[valid={int(np.sum(~np.isnan(out_final)))} px, "
              f"max_score={np.nanmax(out_final) if np.any(~np.isnan(out_final)) else 'N/A'}]")


================ YEAR 2023 ================

Processing 2023-01
  Quarter Q1, bands 1–31 (31 days)
  Loaded 31 daily layers
  Saved: /home/root_1/Documents/CDL/repos/IDS-DRR_Heat/Heat-assam/data_extractor/era5_land/data/monthly_heatday_scores/HEATDAY_2023_01.tif  [valid=1266 px, max_score=24.0]

Processing 2023-02
  Quarter Q1, bands 32–59 (28 days)
  Loaded 28 daily layers
  Saved: /home/root_1/Documents/CDL/repos/IDS-DRR_Heat/Heat-assam/data_extractor/era5_land/data/monthly_heatday_scores/HEATDAY_2023_02.tif  [valid=1266 px, max_score=17.0]

Processing 2023-03
  Quarter Q1, bands 60–90 (31 days)
  Loaded 31 daily layers
  Saved: /home/root_1/Documents/CDL/repos/IDS-DRR_Heat/Heat-assam/data_extractor/era5_land/data/monthly_heatday_scores/HEATDAY_2023_03.tif  [valid=1266 px, max_score=46.0]

Processing 2023-04
  Quarter Q2, bands 1–30 (30 days)
  Loaded 30 daily layers
  Saved: /home/root_1/Documents/CDL/repos/IDS-DRR_Heat/Heat-assam/data_extractor/era5_land/data/monthly_heatday_score

In [ ]:
# summer heatday folders

In [6]:
# =========================================================
# SUMMER HEATDAY SCORE
# Q2 ONLY (APR–JUN)
#
# INPUT:
# quarterly_tiffs_summer2125/
# ├── HI_2021_Q2.tif
# ├── HI_2022_Q2.tif
# ├── HI_2023_Q2.tif
# ├── HI_2024_Q2.tif
# └── HI_2025_Q2.tif
#
# THRESHOLDS:
# AMJ_P80_1990_2023.tif
# AMJ_P88_1990_2023.tif
# AMJ_P95_1990_2023.tif
# AMJ_P99_1990_2023.tif
#
# OUTPUT:
# summer_heatday_scores/
# ├── SUMMER_HEATDAY_2021.tif
# ├── SUMMER_HEATDAY_2022.tif
# ├── SUMMER_HEATDAY_2023.tif
# ├── SUMMER_HEATDAY_2024.tif
# └── SUMMER_HEATDAY_2025.tif
#
# Each output pixel =
# sum of daily heatday scores over Apr-Jun
#
# Daily scoring:
# P80–P88 = 1
# P88–P95 = 2
# P95–P99 = 3
# >=P99   = 4
# =========================================================

import json
from pathlib import Path

import geopandas as gpd
import numpy as np
import rasterio
from rasterio.mask import mask

# =========================================================
# PATHS
# =========================================================

ROOT = Path(
    "/home/root_1/Documents/CDL/repos/IDS-DRR_Heat/Heat-odisha/data_extractor/era5_land"
)

quarter_dir = (
    ROOT
    / "data"
    / "quarterly_tiffs_summer2125"
)

threshold_dir = (
    ROOT
    / "data"
    / "percentile_tiffs"
)

output_dir = (
    ROOT
    / "data"
    / "summer_heatday_scores"
)

output_dir.mkdir(
    parents=True,
    exist_ok=True
)

geojson_path = Path(
    "/home/root_1/Documents/CDL/repos/IDS-DRR_Heat/Heat-odisha/data_extractor/assets/district.geojson"
)

# =========================================================
# LOAD DISTRICT BOUNDARY
# =========================================================

gdf = gpd.read_file(geojson_path)

geoms = json.loads(
    gdf.to_json()
)["features"]

geoms = [
    f["geometry"]
    for f in geoms
]

# =========================================================
# LOAD SINGLE BAND RASTER
# =========================================================

def load_raster(path):

    with rasterio.open(path) as src:
        return src.read(1)

# =========================================================
# LOAD AMJ THRESHOLDS
# =========================================================

print("\nLoading AMJ thresholds...")

p80 = load_raster(
    threshold_dir / "AMJ_P80_1990_2023.tif"
)

p88 = load_raster(
    threshold_dir / "AMJ_P88_1990_2023.tif"
)

p95 = load_raster(
    threshold_dir / "AMJ_P95_1990_2023.tif"
)

p99 = load_raster(
    threshold_dir / "AMJ_P99_1990_2023.tif"
)

print("Thresholds loaded.")

# =========================================================
# PROCESS YEARS
# =========================================================

years = [2021, 2022, 2023, 2024, 2025]

NODATA_VAL = -9999.0

for year in years:

    print(
        f"\n================ {year} ================"
    )

    hi_path = (
        quarter_dir
        / f"HI_{year}_Q2.tif"
    )

    if not hi_path.exists():

        print(
            f"Missing: {hi_path}"
        )

        continue

    # =====================================================
    # LOAD ALL DAILY BANDS
    # =====================================================

    with rasterio.open(hi_path) as src:

        hi = src.read()          # (days, rows, cols)

        meta = src.meta.copy()

        print(
            f"Loaded {src.count} daily layers"
        )

    days, rows, cols = hi.shape

    # =====================================================
    # COMPUTE SUMMER HEATDAY SCORE
    # =====================================================

    summer_score = np.zeros(
        (rows, cols),
        dtype=np.float32
    )

    for d in range(days):

        h = hi[d]

        score = np.zeros(
            (rows, cols),
            dtype=np.uint8
        )

        score[
            (h >= p80)
            & (h < p88)
        ] = 1

        score[
            (h >= p88)
            & (h < p95)
        ] = 2

        score[
            (h >= p95)
            & (h < p99)
        ] = 3

        score[
            h >= p99
        ] = 4

        summer_score += score

    print(
        f"Max score: {np.nanmax(summer_score):.1f}"
    )

    # =====================================================
    # WRITE TEMP FILE
    # =====================================================

    tmp_file = (
        output_dir
        / f"_tmp_{year}.tif"
    )

    meta.update(
        {
            "count": 1,
            "dtype": "float32",
            "nodata": NODATA_VAL
        }
    )

    with rasterio.open(
        tmp_file,
        "w",
        **meta
    ) as dst:

        dst.write(
            summer_score,
            1
        )

    # =====================================================
    # CLIP TO DISTRICT BOUNDARY
    # =====================================================

    with rasterio.open(tmp_file) as src:

        out_image, out_transform = mask(
            src,
            geoms,
            crop=True,
            filled=True,
            nodata=NODATA_VAL,
            all_touched=True
        )

    out_final = out_image[0].astype(
        np.float32
    )

    out_final[
        out_final == NODATA_VAL
    ] = np.nan

    # =====================================================
    # OUTPUT METADATA
    # =====================================================

    out_meta = meta.copy()

    out_meta.update(
        {
            "height": out_image.shape[1],
            "width": out_image.shape[2],
            "transform": out_transform,
            "count": 1,
            "nodata": np.nan
        }
    )

    # =====================================================
    # SAVE FINAL RASTER
    # =====================================================

    out_file = (
        output_dir
        / f"SUMMER_HEATDAY_{year}.tif"
    )

    with rasterio.open(
        out_file,
        "w",
        **out_meta
    ) as dst:

        dst.write(
            out_final,
            1
        )

    tmp_file.unlink()

    valid_pixels = int(
        np.sum(
            ~np.isnan(out_final)
        )
    )

    print(
        f"Saved: {out_file.name}"
    )

    print(
        f"Valid pixels: {valid_pixels}"
    )

    print(
        f"Max score: "
        f"{np.nanmax(out_final):.1f}"
    )

print(
    "\n===================================="
)

print(
    "ALL SUMMER HEATDAY RASTERS CREATED"
)

print(
    "===================================="
)


print(np.nanmin(p80), np.nanmax(p80))
print(np.nanmin(p95), np.nanmax(p95))
print(np.nanmin(p99), np.nanmax(p99))
print(np.nanmin(hi))
print(np.nanmax(hi))


Loading AMJ thresholds...
Thresholds loaded.

================ 2021 ================
Loaded 91 daily layers
Max score: 19.0
Saved: SUMMER_HEATDAY_2021.tif
Valid pixels: 2266
Max score: 19.0

================ 2022 ================
Loaded 91 daily layers
Max score: 60.0
Saved: SUMMER_HEATDAY_2022.tif
Valid pixels: 2266
Max score: 60.0

================ 2023 ================
Loaded 91 daily layers
Max score: 109.0
Saved: SUMMER_HEATDAY_2023.tif
Valid pixels: 2266
Max score: 109.0

================ 2024 ================
Loaded 91 daily layers
Max score: 107.0
Saved: SUMMER_HEATDAY_2024.tif
Valid pixels: 2266
Max score: 107.0

================ 2025 ================
Loaded 91 daily layers
Max score: 28.0
Saved: SUMMER_HEATDAY_2025.tif
Valid pixels: 2266
Max score: 28.0

ALL SUMMER HEATDAY RASTERS CREATED
32.430265990899194 45.807612383759576
33.9422620337075 48.44161070786221
35.07783474678614 50.338609178466065
15.828357824925423
49.83293984101631
